# 05 · Risk map for this week (the demo)

What the technician would open on a Monday morning, for every reservoir in the Jaguaribe system:

- **in:** the most recent clear Sentinel-2 image (last 30 days), the weather of the days before it (NASA POWER), and today's stored volume (Portal Hidrológico);
- **out:** the probability that cyanobacteria are above 20,000 cells/mL, a risk band, and the top 3 reasons in plain language, next to the last lab result COGERH published.

> This project is a prototype. It does not replace laboratory testing of drinking water.

In [1]:
import sys, json
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
import numpy as np
import pandas as pd
import geopandas as gpd
import joblib, folium

sys.path.insert(0, str(Path.cwd().parent / "src"))
from common import PROCESSED, RAW, OUTPUTS, CYANO_THRESHOLD
from satellite import CRS, latest_clear
from weather import grid_cell, fetch_cell, window_features
from fetch_hidro import fetch_volumes, load_db_records
from modeling import time_split
from explain import reasons

TODAY = pd.Timestamp("2026-11-14")
START = (TODAY - pd.Timedelta(days=30)).date()
bundle = joblib.load(OUTPUTS / "model_demo.joblib")  # chosen by the rule in notebook 04b
FEATURES = bundle["features"]
print("model:", bundle["name"], "| alarm threshold:", round(bundle["threshold"], 3))
res = gpd.read_file(PROCESSED / "reservoirs.geojson")
res = res[res["jaguaribe_system"]].set_index("reservoir")
print(len(res), "reservoirs in the Jaguaribe system")

model: Logistic regression (satellite + stored water) | alarm threshold: 0.321
65 reservoirs in the Jaguaribe system


## Latest clear Sentinel-2 image of each reservoir (last 30 days)

In [2]:
aoi_utm = res.to_crs(CRS).geometry
def job(name):
    return {"reservoir": name, **latest_clear(name, aoi_utm[name], res.geometry[name].bounds, START, TODAY.date(),
                                              PROCESSED / "s2_cache")}
with ThreadPoolExecutor(4) as ex:
    sat = pd.DataFrame(list(ex.map(job, res.index))).set_index("reservoir")
print(sat["status"].value_counts().to_string())
print("image dates:", sat["img_date"].value_counts().sort_index().to_dict())

status
ok                          49
no clear image in window    16
image dates: {'2026-10-24': 2, '2026-10-29': 1, '2026-10-31': 2, '2026-11-03': 8, '2026-11-08': 6, '2026-11-10': 12, '2026-11-13': 18}


## Weather and stored volume

Weather windows end the day before the image (NASA POWER runs about two days behind, so a missing last day is filled by the model's training median). Volume comes from the portal's daily-level page for today and for 90 days ago (`volume_change` ≈ change since the previous quarterly campaign, as in training).

In [3]:
live = sat[sat["status"] == "ok"].copy()
live["date"] = pd.to_datetime(live["img_date"])
cells = {n: grid_cell(res.at[n, "lat"], res.at[n, "lon"]) for n in live.index}
series = {c: fetch_cell(*c, RAW / "power") for c in set(cells.values())}
w = pd.DataFrame({n: window_features(series[cells[n]], live.at[n, "date"]) for n in live.index}).T
live = live.join(w)

vol_now = fetch_volumes(PROCESSED.parent.parent, TODAY).set_index("reservoir")
vol_before = fetch_volumes(PROCESSED.parent.parent, TODAY - pd.Timedelta(days=90)).set_index("reservoir")
live["volume_pct"] = vol_now["volume_pct"].reindex(live.index)
live["volume_change"] = live["volume_pct"] - vol_before["volume_pct"].reindex(live.index)
m = live["date"].dt.month
live["month_sin"], live["month_cos"] = np.sin(2 * np.pi * m / 12), np.cos(2 * np.pi * m / 12)
live["rainy_season"] = m.between(2, 5).astype(int)
print("volume found for", int(live["volume_pct"].notna().sum()), "of", len(live))
live[FEATURES].describe().round(3)

volume found for 49 of 49


,NDCI,NDTI,MNDWI,water_frac,volume_pct,volume_change
count,49.000,49.000,49.000,49.000,49.000,49.000
mean,0.058,-0.110,0.166,0.366,49.262,-8.052
std,0.079,0.060,0.093,0.214,30.590,8.016
min,-0.051,-0.287,0.026,0.015,0.440,-33.750
25%,0.008,-0.150,0.091,0.185,20.060,-11.580
50%,0.038,-0.108,0.159,0.326,50.250,-7.620
75%,0.077,-0.055,0.224,0.535,76.530,-4.520
max,0.344,-0.017,0.378,0.798,102.840,18.990


## Risk, band and reasons

- **Alert**: probability ≥ the alarm threshold chosen on validation (the level that still caught 80% of blooms). *Sample this week / start toxin testing.*
- **Lower risk**: below the threshold. *Not "safe": in our test campaigns most of these were still blooms. It can wait for the regular campaign if capacity is short.*

The list is ranked by probability, so the technician starts at the top.

**Known weakness (notebook 04):** in the test campaigns, almost all missed blooms were in the dry-season campaigns (Aug, Nov). This map is for November, so a "Lower risk" pin deserves extra caution.

In [4]:
live["risk"] = bundle["model"].predict_proba(live[FEATURES])[:, 1]
thr = bundle["threshold"]
live["band"] = np.where(live["risk"] >= thr, "Alert", "Lower risk")
feat_all = pd.read_csv(PROCESSED / "features.csv", parse_dates=["date"])
train_means = feat_all[time_split(feat_all) == "train"][FEATURES].mean()
live["reasons"] = reasons(bundle, live, train_means)

## Last lab result published by COGERH

From the portal database (its newest campaign is Aug/2026). This is what the technician knows today without the tool. Values below 1,000 with decimals are flagged: the database sometimes stores counts divided by 1,000 (we found this in Feb/2020 and Nov/2021).

In [5]:
db = load_db_records(PROCESSED.parent.parent).sort_values("date_db")
last = db.groupby("reservoir").tail(1).set_index("reservoir")
live["last_lab_date"] = last["date_db"].reindex(live.index).dt.date
live["last_lab_cyano"] = last["cyano_db"].reindex(live.index)
odd = (live["last_lab_cyano"] < 1000) & (live["last_lab_cyano"] % 1 != 0)
live["last_lab_note"] = np.where(odd, "value looks mis-entered in the database", "")
live["last_lab_class"] = np.where(live["last_lab_cyano"].isna() | odd, "unknown",
                                  np.where(live["last_lab_cyano"] > CYANO_THRESHOLD, "above 20,000", "below 20,000"))

## Ranked list

**Castanhão** is highlighted: in Sep/2019 it was the only source for five Jaguaribe Valley cities: *"Atualmente, apenas cinco recebem água do reservatório: Tabuleiro do Norte, São João do Jaguaribe, Limoeiro do Norte, Jaguaribara e Russas"* (Diário do Nordeste, 9 Sep 2019). Supply arrangements change, so this is context, not a current operational fact.

In [6]:
live["name"] = res["name"].reindex(live.index)
live["basin"] = res["basin"].reindex(live.index)
table = live.sort_values("risk", ascending=False)[["name", "basin", "risk", "band", "img_date", "volume_pct",
                                                   "last_lab_date", "last_lab_cyano", "last_lab_class", "reasons"]]
table.assign(reasons=table["reasons"].map(" | ".join)).to_csv(OUTPUTS / "risk_table.csv")
no_image = sat[sat["status"] != "ok"].index.tolist()
print("no clear image in the last 30 days:", [res.at[n, "name"] for n in no_image])
table.head(20).style.format({"risk": "{:.2f}", "volume_pct": "{:.0f}", "last_lab_cyano": "{:,.0f}"})

no clear image in the last 30 days: ['Facundo', 'Faé', 'Figueiredo', 'Gomes', 'Jenipapeiro II', 'João Luís', 'Manoel Balbino', 'Nova Floresta', 'Pau Preto', 'Riacho da Serra', 'São Domingos II', 'São José I', 'Tatajuba', 'Tigre', 'José Barbosa Filho (Umari)', 'Valério']


,name,basin,risk,band,img_date,volume_pct,last_lab_date,last_lab_cyano,last_lab_class,reasons
reservoir,,,,,,,,,,
monsenhor tabosa,Monsenhor Tabosa,Banabuiú,0.93,Alert,2026-11-08,47,2026-08-20,"369,881","above 20,000","['↑ green water (chlorophyll index 0.344)', '↑ clear water (turbidity index -0.23)', '↓ volume changed -8 points in 3 months']"
canafistula,Canafístula,Médio Jaguaribe,0.84,Alert,2026-11-13,5,2026-08-04,"304,400","above 20,000","['↑ green water (chlorophyll index 0.278)', '↓ turbid water (turbidity index -0.13)', '↓ volume changed -5 points in 3 months']"
rivaldo de carvalho,Rivaldo de Carvalho,Alto Jaguaribe,0.82,Alert,2026-11-10,49,2026-08-25,"219,451","above 20,000","['↑ green water (chlorophyll index 0.213)', '↑ 49% of the area near the sampling point is water', '↓ volume changed -10 points in 3 months']"
vieirao,Vieirão,Banabuiú,0.79,Alert,2026-11-10,25,2026-07-16,"261,925","above 20,000","['↑ green water (chlorophyll index 0.228)', '↑ 35% of the area near the sampling point is water', '↓ turbid water (turbidity index -0.13)']"
potiretama,Potiretama,Médio Jaguaribe,0.76,Alert,2026-11-10,11,2026-08-11,"273,889","above 20,000","['↑ 76% of the area near the sampling point is water', '↓ little chlorophyll seen (index 0.095)', '↓ volume changed -5 points in 3 months']"
patu,Patu,Banabuiú,0.74,Alert,2026-11-10,73,2026-08-06,"272,544","above 20,000","['↑ 80% of the area near the sampling point is water', '↓ little chlorophyll seen (index 0.065)', '↑ clear water (turbidity index -0.22)']"
oros,Orós,Alto Jaguaribe,0.71,Alert,2026-11-13,85,2026-08-20,"138,042","above 20,000","['↑ 69% of the area near the sampling point is water', '↓ little chlorophyll seen (index 0.083)', '↓ volume changed -12 points in 3 months']"
arneiroz ii,Arneiroz II,Alto Jaguaribe,0.70,Alert,2026-11-08,82,2026-07-14,"371,651","above 20,000","['↑ 58% of the area near the sampling point is water', '↓ volume changed -12 points in 3 months', '↓ little chlorophyll seen (index 0.112)']"
curral velho,Curral Velho,Banabuiú,0.66,Alert,2026-11-13,91,2026-08-05,"484,916","above 20,000","['↑ 67% of the area near the sampling point is water', '↓ little chlorophyll seen (index 0.065)', '↓ turbid water (turbidity index -0.10)']"


## Map

In [7]:
colors = {"Alert": "#c0392b", "Lower risk": "#2e86c1"}
fmap = folium.Map(location=[res["lat"].mean(), res["lon"].mean()], zoom_start=8, tiles="CartoDB positron")
for n, r in live.iterrows():
    special = "<br><b>Supplied Tabuleiro do Norte, Limoeiro do Norte, Russas, Jaguaribara and São João do Jaguaribe (Diário do Nordeste, Sep 2019)</b>" if n == "castanhao" else ""
    last_txt = (f"{r['last_lab_cyano']:,.0f} cells/mL on {r['last_lab_date']} {r['last_lab_note']}"
                if pd.notna(r["last_lab_cyano"]) else "none")
    html = (f"<b>{r['name']}</b> ({r['basin']})<br>Risk of &gt; 20,000 cells/mL: <b>{r['risk']:.0%}</b> "
            f"(<span style='color:{colors[r['band']]}'><b>{r['band']}</b></span>)<br>"
            f"Image: {r['img_date']} · Volume: {r['volume_pct']:.0f}%<br>"
            + "<br>".join(r["reasons"]) + f"<br><i>Last lab result: {last_txt}</i>{special}")
    folium.CircleMarker([res.at[n, "lat"], res.at[n, "lon"]], radius=11 if n == "castanhao" else 7,
                        color="black" if n == "castanhao" else colors[r["band"]], weight=3 if n == "castanhao" else 1,
                        fill=True, fill_color=colors[r["band"]], fill_opacity=0.85,
                        popup=folium.Popup(html, max_width=340), tooltip=f"{r['name']}: {r['risk']:.0%}").add_to(fmap)
for n in no_image:
    folium.CircleMarker([res.at[n, "lat"], res.at[n, "lon"]], radius=5, color="gray", fill=True, fill_opacity=0.4,
                        tooltip=f"{res.at[n, 'name']}: no clear image in the last 30 days").add_to(fmap)
legend = ("<div style='position:fixed;bottom:20px;left:20px;z-index:999;background:white;padding:8px;border:1px solid #999;font-size:12px'>"
          "<b>Alerta Açude</b> · " + TODAY.strftime("%d %b %Y") + "<br>"
          + "".join(f"<span style='color:{c}'>●</span> {b}<br>" for b, c in colors.items())
          + "<span style='color:gray'>●</span> no clear image<br><i>Dry season: the model under-warns (see README).<br>Prototype. Does not replace laboratory testing.</i></div>")
fmap.get_root().html.add_child(folium.Element(legend))
fmap.save(str(OUTPUTS / "risk_map.html"))
fmap

C:\Users\Filipe\miniconda3\envs\alerta\Lib\site-packages\folium\raster_layers.py:130: UserWarning: CartoDB tiles now require an API key. Please provide one to continue using the tiles. You can request the key at https://carto.com/basemaps/apikey/.
  tiles = tiles.build_url(fill_subdomain=False, scale_factor="{r}")  # type: ignore
